<a href="https://colab.research.google.com/github/MElsdk-lab/Biochar_forest_estimation/blob/main/Notebook_2_Forest_Area_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# NOTEBOOK 2 — Forest Area Analysis & FAO Comparison
# University of Pittsburgh | Biochar Feedstock Methodology
# ============================================================

In [2]:
# ── CELL 1: Install Libraries ─────────────────────────────────────────────────
!pip install -q pandas matplotlib
print('✅ Libraries installed')

✅ Libraries installed


In [13]:
# ── CELL 2: clone repo if not already cloned ─────────────────────
import os
import getpass
import subprocess

if not os.path.exists('/content/Biochar_forest_estimation'):
    PAT = getpass.getpass('Enter PAT: ')
    !git config --global user.email "sdkmajd@gmail.com"
    !git config --global user.name "MElsdk-lab"
    subprocess.run(
        f'git clone https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git',
        shell=True
    )

%cd /content/Biochar_forest_estimation/
!git fetch origin
!git reset --hard origin/main

DATA_folder = '/content/Biochar_forest_estimation/data/'

print('✅ Ready')

/content/Biochar_forest_estimation
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 9 (delta 6), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 2.17 KiB | 556.00 KiB/s, done.
From https://github.com/MElsdk-lab/Biochar_forest_estimation
   e921383..ad1ffa6  main       -> origin/main
HEAD is now at ad1ffa6 Created using Colab
✅ Ready


In [4]:
## ── CELL 3: import Libraries and results & data from GEE and FOA ─────────────────────────────────────────────────

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

from data_config import (build_country_lookup, FAO_LSIB_REGION, country_thresholds, state_thresholds,
                         fao_fra_2025_raw_data,
                         americas_region, europe_region, africa_region, asia_region, near_east_region)


print('✅ Libraries imported')
print('✅ Data config loaded')


✅ Libraries imported
✅ Data config loaded


In [5]:
## ── CELL 4: establish total countries and US states results & data ─────────────────────────────────────────────────

#countries
countries_files = [os.path.join(DATA_folder, f"forest_area_{i}0.csv") for i in range(1, 6)]

country_total_forest_area = pd.concat([pd.read_csv(f) for f in countries_files],ignore_index=True)

country_total_forest_area=country_total_forest_area.rename(columns = {'country_na' :'country', 'sum' : 'area_Mha'})
country_total_forest_area.sort_values(by= ['country', 'threshold'], ascending=True, inplace=True)
country_total_forest_area=country_total_forest_area.groupby(['country', 'threshold'])['area_Mha'].sum().reset_index()

country_total_forest_area.to_csv(DATA_folder + 'country_total_forest_area.csv', index=False)

print(f'✅ Forest data loaded and saved with {int(len(country_total_forest_area)/len(country_thresholds))} rows')
#States

states_files = [os.path.join(DATA_folder, f"states_forest_area_{i}0.csv") for i in range(1, 6)]

states_total_forest_area = pd.concat([pd.read_csv(f) for f in states_files], ignore_index=True)

states_total_forest_area = states_total_forest_area.rename(columns={'NAME': 'state', 'sum': 'area_Mha'})
states_total_forest_area.sort_values(by=['state', 'threshold'], ascending=True, inplace=True)
states_total_forest_area = states_total_forest_area.groupby(['state', 'threshold'])['area_Mha'].sum().reset_index()
states_total_forest_area.to_csv(DATA_folder + 'states_total_forest_area.csv', index=False)

alaska_state_forest_area = states_total_forest_area[states_total_forest_area['state' ]== 'Alaska']
hawaii_state_forest_area = states_total_forest_area[states_total_forest_area['state' ]== 'Hawaii']


print(f'✅ Forest data loaded and saved with {int(len(states_total_forest_area)/len(state_thresholds))} rows')


✅ Forest data loaded and saved with 195 rows
✅ Forest data loaded and saved with 50 rows


In [20]:
## ── CELL 5 : build region and subregion info for countries in 'country_total_forest_area' file by the intermidate of FAO_LSIB_REGION   ─────────────────────────────────────────────────
# 1. Generate the lookup dictionary
country_lookup = build_country_lookup(FAO_LSIB_REGION)

# 2. Create the new columns by looking up each country name
# we could actualy transform the country_lookup from a dictionary to a data frame then merge with df bsed onthe country name
country_total_forest_area  ['region'] = country_total_forest_area['country'].map(lambda l: country_lookup.get(l, {}).get('region', 'unknown'))
country_total_forest_area  ['subregion'] = country_total_forest_area['country'].map(lambda l: country_lookup.get(l, {}).get('subregion', 'unknown'))

print(f"✅countries are affilited to regions and subregions")

✅countries are affilited to regions and subregions


In [19]:
# ── CELL 6: Merge GEE results with FAO 2000 ──────────────
FAO_fra_2025_data = pd.read_csv(DATA_folder + 'fao_fra_2025_data.csv')

FAO_forest_area_country_2000 = FAO_fra_2025_data[['country', '2000_area_Mha']]

gee_fao_comparison = country_total_forest_area.merge(FAO_forest_area_country_2000, on='country', how='left')

#print(gee_fao_comparison.head(40))

print(f"Results from GEE has {len(country_total_forest_area)}")

print(f"FAO has {len(FAO_forest_area_country_2000)}")
#counting th number of countries with data; since there is 5  different thresholds, nunique returns the number of unique values in the selected column:
print(f'Total countries: {gee_fao_comparison["country"].nunique()}')

# if mising, unique show the the values of unique values
missing = gee_fao_comparison[gee_fao_comparison['2000_area_Mha'].isna()]['country'].unique()
print(f"Total countries with no FAO match: {len(missing)}")
print('GEE results are merged with FAO 2000 ')



Results from GEE has 975
FAO has 195
Total countries: 195
Total countries with no FAO match: 0
GEE results are merged with FAO 2000 


In [ ]:
# ── CELL 7: aggregate by subregion and region. ──────────────

gee_fao_comparison_subregion = gee_fao_comparison.groupby(['subregion', 'threshold']).agg(
    region=('region', 'first'), # Ensure 'region' is carried over
    subregion_total_forest_area=('area_Mha', 'sum'),
    FAO_2000_Mha=('FAO_2000_Mha', 'sum')
).reset_index()

gee_fao_comparison_region = gee_fao_comparison_subregion.groupby(['region', 'threshold']).agg(
    region_total_forest_area=('subregion_total_forest_area', 'sum'),
    FAO_2000_Mha=('FAO_2000_Mha', 'sum')
).reset_index()

gee_fao_comparison_world = gee_fao_comparison_region.groupby(['threshold']).agg(
    world_total_forest_area=('region_total_forest_area', 'sum'),
    FAO_2000_Mha=('FAO_2000_Mha', 'sum')
).reset_index()

gee_fao_comparison_subregion.to_csv(DATA_folder + 'gee_fao_comparison_subregion.csv', index=False)
gee_fao_comparison_region.to_csv(DATA_folder + 'gee_fao_comparison_region.csv', index=False)
gee_fao_comparison_world.to_csv(DATA_folder + 'gee_fao_comparison_world.csv', index=False)

print('✅ Results are aggregated by subregion and region')

In [ ]:
# ── CELL 8: plot GEE vs FAO. ──────────────
#create the figure and axes

# add a devition column in gee_fao_comparison
gee_fao_comparison['deviation_Mha'] = gee_fao_comparison['area_Mha'] - gee_fao_comparison['FAO_2000_Mha']
gee_fao_comparison['deviation_%'] = abs((gee_fao_comparison['deviation_Mha'] / gee_fao_comparison['FAO_2000_Mha'])/gee_fao_comparison['FAO_2000_Mha']).multiply(100)
clean_dev = gee_fao_comparison.replace([float('inf'), float('-inf')], float('nan'))
#print(f"Mean deviation: {clean_dev['deviation_%'].median():.2f}%")

gee_fao_comparison.to_csv(DATA_folder + 'gee_fao_comparison.csv', index=False)


fig, ax = plt.subplots(figsize=(10, 8))
plt.xlim(right = 1.05 *max(gee_fao_comparison['FAO_2000_Mha']))
plt.ylim(top = 1.05 *max(gee_fao_comparison['FAO_2000_Mha']))

#plot the dots
for threshold in country_thresholds:
  df =gee_fao_comparison[gee_fao_comparison['threshold'] == threshold]
  cleandev_df= clean_dev[clean_dev['threshold'] == threshold]
  x_values = df['FAO_2000_Mha'].dropna()   #check where are the nan in both column
  y_values = df['area_Mha'].dropna()

  ax.scatter(x_values, y_values, label='GEE 10%')
  slope, intercept, r_value, p_value, std_err = stats.linregress(x_values, y_values)
  print(f'for threshold {threshold}%, R² = {r_value**2:.3f}, average deviation = {cleandev_df["deviation_%"].median():.2f}%')




#add labels, limit and title
ax.set_xlabel('FAO 2000 (Mha)')
ax.set_ylabel('GEE forest are (Mha)')
ax.set_title('GEE vs FAO forest area comparison')

match_max=max(gee_fao_comparison['FAO_2000_Mha'])

ax.plot([0, match_max], [0,match_max], linestyle = '--', color = 'black')

#show the plot
plt.show()

In [ ]:
# ── push results to GitHub ────────────────────────────────
import subprocess

# ask for PAT only if not already defined
if 'PAT' not in dir():
    import getpass
    PAT = getpass.getpass('Enter PAT: ')

%cd /content/Biochar_forest_estimation/
!git add .
!git commit -m "update results"

result = subprocess.run(
    f'git push https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git main',
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print('✅ Pushed to GitHub')
else:
    print('❌ Push failed')
    print(result.stderr)